# Basic Transfer Learning Example with MNIST and MobileNetV2


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
print(tf.__version__)

2.19.0


## Load MNIST dataset

In [2]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
print(train_images.shape, train_labels.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28) (60000,)


## Preprocess MNIST to match MobileNet input requirements

In [3]:
def preprocess_mnist(image, label):
    image = tf.expand_dims(image, axis=-1)          # (28,28,1)
    image = tf.image.resize(image, (96, 96))        # resize
    image = tf.image.grayscale_to_rgb(image)        # (96,96,3)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_ds = train_ds.map(preprocess_mnist).batch(64)

test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_ds = test_ds.map(preprocess_mnist).batch(64)

## Load pretrained MobileNetV2 and freeze it

In [4]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Add a simple classification head

In [5]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Compile and train the model

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, epochs=2)

Epoch 1/2
938/938 ━━━━━━━━━━━━━━━━━━━━ 398s 418ms/step - accuracy: 0.8607 - loss: 0.4650
Epoch 2/2
938/938 ━━━━━━━━━━━━━━━━━━━━ 382s 407ms/step - accuracy: 0.9641 - loss: 0.1139


## Evaluate on test set

In [7]:
test_loss, test_acc = model.evaluate(test_ds)
print('Test accuracy:', test_acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 64s 399ms/step - accuracy: 0.9588 - loss: 0.1227
Test accuracy: 0.9656000137329102
